# Price-History Filter

Keeps the price-history and price-summary sheets consistent with the final product list.
The `products` sheet defines the confirmed set of seller item IDs (`vendorItemId`); rows in
`price_history` and `price_summary` whose ID is not in that set are dropped. A
market-summary sheet, which is not per-product, is passed through unchanged.

> See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).

In [ ]:
import os
from datetime import datetime
import pandas as pd

# Sheet names written by notebooks 01-02 (Korean, as produced by the pipeline)
PRODUCTS, HISTORY, SUMMARY = "제품분석", "가격이력", "가격요약"   # products / price history / price summary
ID_COL = "vendorItemId"

try:
    from google.colab import files
    INPUT = list(files.upload().keys())[0]
except ImportError:
    INPUT = "../sample-data/coupang_workbook.xlsx"

xls = pd.ExcelFile(INPUT)
sheets = {s: pd.read_excel(INPUT, sheet_name=s) for s in xls.sheet_names}
print("sheets:", xls.sheet_names)


def find_id(df, sheet):
    for c in df.columns:
        if str(c).strip().lower() == ID_COL.lower():
            return c
    raise ValueError(f"[{sheet}] no {ID_COL} column: {list(df.columns)}")


ids = lambda s: s.dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

valid = set(ids(sheets[PRODUCTS][find_id(sheets[PRODUCTS], PRODUCTS)]))
valid.discard("")
print(f"confirmed products: {len(valid)}")

for sheet in (HISTORY, SUMMARY):
    df = sheets[sheet]
    col = find_id(df, sheet)
    key = ids(df[col].fillna(""))
    before_rows, before_ids = len(df), set(key) - {""}
    sheets[sheet] = df[key.isin(valid)].copy()
    after_ids = set(ids(sheets[sheet][col])) - {""}
    print(f"[{sheet}] rows {before_rows:,} -> {len(sheets[sheet]):,} | "
          f"products {len(before_ids)} -> {len(after_ids)} | "
          f"missing vs product list: {len(valid - after_ids)}")

out = f"{os.path.splitext(os.path.basename(INPUT))[0]}_filtered_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as w:
    for s in xls.sheet_names:                  # keep original sheet order
        sheets[s].to_excel(w, sheet_name=s, index=False)
print("saved:", out)